# Dataplex: Metadata Change Feeds Demo (February 2026 Preview)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/POH/blob/main/dataplex_change_feed_demo.ipynb)

This notebook demonstrates how to programmatically configure **Metadata Change Feeds** in Dataplex. This feature provides real-time Pub/Sub notifications for metadata events like schema changes.

## Use Case
A Data Governance team needs to trigger a 'Data Quality' check immediately after a table's schema is updated in BigQuery. We'll use the Dataplex API to enable the feed.

### Requirements
- Dataplex Lake and Asset configured.
- Pub/Sub Topic and Subscription.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import dataplex_v1
from google.cloud import pubsub_v1
import os

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
lake_id = 'my-lake'
topic_id = 'metadata-change-feed-topic'

### 2. Configure the Change Feed Metadata

In the February 2026 release, the `metastore_config` for a Lake includes a dedicated `change_feed_config`.

In [ ]:
# Example configuration for enabling the feed
lake_patch = {
    "metastore_config": {
        "change_feed_config": {
            "enabled": True,
            "pubsub_topic": f"projects/{project_id}/topics/{topic_id}"
        }
    }
}

print(f"Enabling Metadata Change Feed for Lake: {lake_id}...")
print(f"Target Topic: {lake_patch['metastore_config']['change_feed_config']['pubsub_topic']}")
print("Success: Lake metadata update prepared.")

### 3. Process Metadata Notifications

When a schema change occurs, a structured JSON message is published to the topic.

In [ ]:
import json

# Simulated notification processing logic
def handle_metadata_event(message_data):
    event = json.loads(message_data)
    print(f"--- Metadata Event Detected ---")
    print(f"Type: {event['event_type']}")
    print(f"Resource: {event['resource']}")
    print(f"Changes: {', '.join(event['change_details']['added_fields'])}")
    print(f"Action: Triggering DataQualityCheck_V2...")

mock_message = json.dumps({
  "event_type": "METADATA_SCHEMA_UPDATE",
  "resource": "projects/my-project/datasets/sales/tables/transactions",
  "change_details": {
    "added_fields": ["discount_code", "referral_id"],
    "timestamp": "2026-03-05T14:30:00Z"
  }
})

handle_metadata_event(mock_message)

### 4. Things to remember or know
- **Real-Time Data Governance**: No more polling for schema changes. Automate data quality and security checks instantly.
- **Event-Driven AI**: Use metadata changes to trigger the re-training of models or update the knowledge base of an agent.
- **Full Lineage Integration**: Change feeds are integrated with Dataplex Lineage to track the impact of metadata updates downstream.